# Training YOLOv8 Smart Gate

Notebook ini memakai `dataset_merged/` sebagai dataset utama. `plat-nomor/` digunakan sebagai fallback jika dataset gabungan tidak tersedia.

In [ ]:
!pip install -q ultralytics pyyaml

from pathlib import Path

# Colab extension VS Code memakai workspace lokal; tidak perlu upload ZIP.
HERE = Path.cwd().resolve()
SEARCH_ROOTS = [
    HERE,
    HERE / 'prototype',
    HERE.parent,
    HERE.parent.parent,
    Path('/content'),
]
candidates = []
for root in SEARCH_ROOTS:
    candidates.extend([root / 'dataset_merged', root / 'plat-nomor'])
DATA_ROOT = next((path for path in candidates if (path / 'data.yaml').is_file()), None)
if DATA_ROOT is None:
    raise FileNotFoundError(
        'Dataset lokal tidak ditemukan. Pastikan prototype/dataset_merged/data.yaml '
        'atau prototype/plat-nomor/data.yaml tersedia.'
    )
DATA_YAML = DATA_ROOT / 'data.yaml'
print('Workspace:', HERE)
print('Dataset lokal:', DATA_ROOT)
print('Config:', DATA_YAML)

In [ ]:
import yaml

config = yaml.safe_load(DATA_YAML.read_text())
print(config)
assert config.get('nc') == 1, 'Dataset harus memiliki satu kelas.'
assert config.get('names') == ['number-plate'], 'Kelas harus number-plate.'
for split in ('train', 'val'):
    split_path = (DATA_ROOT / config[split]).resolve()
    if not split_path.is_dir():
        raise FileNotFoundError(f'{split} tidak ditemukan: {split_path}')
print('Struktur dataset valid.')

In [ ]:
def count_labels(folder):
    files = list(Path(folder).glob('*.txt'))
    objects = sum(len(path.read_text().splitlines()) for path in files)
    return len(files), objects

for split in ('train', 'valid', 'test'):
    label_dir = DATA_ROOT / split / 'labels'
    if label_dir.is_dir():
        files, objects = count_labels(label_dir)
        print(f'{split}: {files} file label, {objects} anotasi')
    else:
        print(f'{split}: tidak tersedia')

## Training

Gunakan GPU Colab: **Runtime > Change runtime type > T4 GPU**. Kotak anotasi harus tepat mengelilingi papan plat, bukan bumper, kamera, gril, atau seluruh kendaraan.

In [ ]:
from ultralytics import YOLO
import torch

device = 0 if torch.cuda.is_available() else 'cpu'
batch = 16 if device != 'cpu' else 4
print('Device:', device)

RUN_DIR = DATA_ROOT.parent / 'runs' / 'detect'
model = YOLO('yolov8n.pt')
model.train(
    data=str(DATA_YAML),
    epochs=50,
    imgsz=640,
    batch=batch,
    device=device,
    project=str(RUN_DIR),
    name='smartgate-colab',
    patience=12,
    hsv_h=0.015,
    hsv_s=0.55,
    hsv_v=0.35,
    degrees=5.0,
    translate=0.08,
    scale=0.35,
    fliplr=0.0,
    mosaic=0.7,
    mixup=0.05,
    close_mosaic=10,
    pretrained=True,
    plots=True,
    val=True,
)
print('Training selesai.')

In [ ]:
best_pt = RUN_DIR / 'smartgate-colab' / 'weights' / 'best.pt'
if not best_pt.is_file():
    raise FileNotFoundError(best_pt)
trained = YOLO(str(best_pt))
metrics = trained.val(data=str(DATA_YAML), imgsz=640, device=device)
print('best.pt:', best_pt)
print(f'mAP50: {metrics.box.map50:.4f}')
print(f'mAP50-95: {metrics.box.map:.4f}')

In [ ]:
print('Bobot selesai:', best_pt)
print('Salin best.pt ke prototype/runs/detect/smartgate-colab/weights/best.pt')

Setelah training selesai, salin `best.pt` ke:

`Tugas Citra Digital/prototype/runs/detect/smartgate-colab/weights/best.pt`

Lalu jalankan aplikasi dengan `SGS_YOLO_WEIGHTS` menunjuk ke file tersebut dan restart Flask.